### Линейные опорные модели.

In [28]:
import sys
sys.path.append('..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.linear_model import LinearRegression, Ridge
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

from evaluate_sku import split_history, compute_wape, build_evaluation, compute_mase_per_sku

In [2]:
train = pd.read_csv('../data/train.csv')
cutoff = '2021-08'
history, actuals = split_history(train, cutoff=cutoff)

In [7]:
history

,sku,unit,2019-01,2019-02,2019-03,2019-04,2019-05,2019-06,2019-07,2019-08,...,2020-11,2020-12,2021-01,2021-02,2021-03,2021-04,2021-05,2021-06,2021-07,2021-08
0,0,ШТ,128.0,125.0,179.0,199.0,204.0,246.0,172.0,235.0,...,175.0,271.0,335.0,379.0,250.0,317.0,172.0,193.0,173.0,121.0
1,1,ШТ,65.0,64.0,54.0,91.0,74.0,91.0,138.0,115.0,...,85.0,80.0,47.0,68.0,65.0,72.0,82.0,85.0,69.0,75.0
2,2,ШТ,157.0,205.0,210.0,205.0,265.0,248.0,292.0,268.0,...,350.0,327.0,333.0,334.0,380.0,409.0,309.0,388.0,438.0,404.0
3,3,ШТ,2450.0,2280.0,2144.0,2283.0,2177.0,2888.0,3066.0,2288.0,...,1642.0,1625.0,2537.0,2806.0,2085.0,1984.0,2086.0,2871.0,2204.0,1821.0
4,4,ШТ,119.0,154.0,154.0,270.0,513.0,718.0,720.0,292.0,...,392.0,599.0,221.0,222.0,258.0,289.0,662.0,759.0,616.0,365.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1651,1651,ШТ,536.0,920.0,1503.0,3268.0,3878.0,6611.0,8629.0,6499.0,...,1551.0,1506.0,1857.0,1482.0,1850.0,2930.0,5484.0,12128.0,6177.0,6953.0
1652,1652,ПК,138.0,247.0,255.0,2033.0,1268.0,476.0,851.0,643.0,...,198.0,258.0,110.0,255.0,244.0,2431.0,1096.0,496.0,523.0,586.0
1653,1653,ШТ,708.0,730.0,828.0,959.0,918.0,908.0,1049.0,1094.0,...,1165.0,1526.0,1074.0,941.0,1353.0,1167.0,1035.0,1025.0,1158.0,1258.0
1654,1654,ШТ,594.0,654.0,820.0,740.0,697.0,690.0,1006.0,703.0,...,934.0,968.0,667.0,888.0,1226.0,1133.0,802.0,893.0,860.0,973.0


In [6]:
actuals

,id,actual
0,0_2021-09,318.0
1,1_2021-09,100.0
2,2_2021-09,293.0
3,3_2021-09,2187.0
4,4_2021-09,321.0
...,...,...
4963,1651_2021-11,1672.0
4964,1652_2021-11,160.0
4965,1653_2021-11,2129.0
4966,1654_2021-11,1171.0


In [8]:
# Собираем матрицу признаков X
X = pd.DataFrame({
    'lag_1': history['2021-08'],
    'lag_12': history['2020-09'],
    'mean_12': history.loc[:, '2020-09':'2021-08'].mean(axis=1)
})
 
X

,lag_1,lag_12,mean_12
0,121.0,221.0,228.666667
1,75.0,108.0,78.583333
2,404.0,251.0,357.000000
3,1821.0,2314.0,2211.500000
4,365.0,345.0,425.000000
...,...,...,...
1651,6953.0,5913.0,4305.500000
1652,586.0,774.0,611.083333
1653,1258.0,1163.0,1176.000000
1654,973.0,625.0,893.666667


In [10]:
# 1. То, что мы пытаемся угадать на тренировке (продажи в августе 2021)
y_train = history['2021-08']
# 2. Признаки, которые были известны до августа 2021
X_train = pd.DataFrame({
    'lag_1': history['2021-07'],
    'lag_12': history['2020-08'],
    'mean_12': history.loc[:, '2019-09':'2020-08'].mean(axis=1)
})

In [14]:
# 1. Обычная регрессия без штрафа
lin_reg = LinearRegression()
lin_reg.fit(X_train, y_train)
# 2. Ridge со штрафом
ridge_reg = Ridge(alpha=100)
ridge_reg.fit(X_train, y_train)
print("Веса без штрафа (LinearRegression):", lin_reg.coef_)
print("Веса со штрафом (Ridge):           ", ridge_reg.coef_)

Веса без штрафа (LinearRegression): [ 0.86244403 -0.04778514  0.24366465]
Веса со штрафом (Ridge):            [ 0.86244403 -0.04778514  0.24366465]


In [16]:
# 1. Обычная регрессия со скейлером
lin_pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('model', LinearRegression())
])
lin_pipe.fit(X_train, y_train)
# 2. Ridge со скейлером и разной силой штрафа
ridge_pipe_10 = Pipeline([
    ('scaler', StandardScaler()),
    ('model', Ridge(alpha=10.0))
])
ridge_pipe_10.fit(X_train, y_train)
ridge_pipe_100 = Pipeline([
    ('scaler', StandardScaler()),
    ('model', Ridge(alpha=100.0))
])
ridge_pipe_100.fit(X_train, y_train)
print("Веса без штрафа:             ", lin_pipe.named_steps['model'].coef_)
print("Веса Ridge (слабый штраф 10):", ridge_pipe_10.named_steps['model'].coef_)
print("Веса Ridge (сильный штраф 100):", ridge_pipe_100.named_steps['model'].coef_)

Веса без штрафа:              [14400.52470636  -834.75798298  3006.0712945 ]
Веса Ridge (слабый штраф 10): [14102.59069938  -692.14880318  3120.92179128]
Веса Ridge (сильный штраф 100): [12079.48852323   340.82835053  3764.50130619]


теперь пробуем предсказывать на настоящих данных пайплайн с регуляризацией с alpha = 100 (сентябрь - ноябрь 2021)

Сентябрь

In [22]:
X_test_sep = pd.DataFrame(
    {
        'lag_1': history['2021-08'],
        'lag_12': history['2020-09'],
        'mean_12': history.loc[:, '2020-09':'2021-08'].mean(axis=1)
    }
)

pred_sep_100 = ridge_pipe_100.predict(X_test_sep)

Октябрь

In [23]:
X_test_oct = pd.DataFrame(
    {
        'lag_1': history['2021-08'],
        'lag_12': history['2020-10'],
        'mean_12': history.loc[:, '2020-09':'2021-08'].mean(axis=1)
    }
)

pred_oct_100 = ridge_pipe_100.predict(X_test_oct)

Ноябрь

In [24]:
X_test_nov = pd.DataFrame(
    {
        'lag_1': history['2021-08'],
        'lag_12': history['2020-11'],
        'mean_12': history.loc[:, '2020-09':'2021-08'].mean(axis=1)
    }
)

pred_nov_100 = ridge_pipe_100.predict(X_test_nov)

Собираем все в один датасет, чтобы посчитать метрики 

In [26]:
sub_sep = pd.DataFrame({'id': history['sku'].astype(str) + '_2021-09',
                        'prediction': pred_sep_100})
sub_oct = pd.DataFrame({'id': history['sku'].astype(str) + '_2021-10',
                        'prediction': pred_oct_100})
sub_nov = pd.DataFrame({'id': history['sku'].astype(str) + '_2021-11',
                        'prediction': pred_nov_100})

sub_all = pd.concat([sub_sep, sub_oct, sub_nov], ignore_index=True)
sub_all['prediction'] = sub_all['prediction'].clip(lower=0)

sub_all

,id,prediction
0,0_2021-09,313.768326
1,1_2021-09,232.488913
2,2_2021-09,558.246289
3,3_2021-09,2189.491824
4,4_2021-09,552.615798
...,...,...
4963,1651_2021-11,6526.253158
4964,1652_2021-11,766.409166
4965,1653_2021-11,1443.805239
4966,1654_2021-11,1146.967065


In [29]:
# Стыкуем прогноз с реальным экзаменом (actuals) строго по 'id'
eval_df = build_evaluation(actuals, sub_all)
# 1. Считаем главную лидербордную метрику WAPE
wape_ridge = compute_wape(eval_df['actual'].to_numpy(), eval_df['prediction'].to_numpy())
# 2. Считаем метрику MASE для отчёта
mase_ridge = compute_mase_per_sku(history, eval_df)

print(f"WAPE нашей Ridge-модели: {wape_ridge:.4f}")
print(f"Медиана MASE:           {mase_ridge.median():.4f}")
print(f"Доля SKU с MASE < 1:    {(mase_ridge < 1).mean():.1%}")

WAPE нашей Ridge-модели: 0.3621
Медиана MASE:           1.8621
Доля SKU с MASE < 1:    26.8%


Обучили базовую Ridge-регрессию на признаках лагов и среднего. Модель получила WAPE = 0.3621, что лучше бейзлайна последнего месяца (0.380). Однако модель уступила среднегодовому уровню (0.275), так как обучалась только на пиковом августе и не учитывала осеннее падение спроса. Для улучшения требуется обучение на нескольких сезонах и учёт календаря.